# LangGraph Agentic Workflow: Product Description Generator

This notebook demonstrates how to build an **agentic workflow** using LangGraph.

Instead of asking one LLM prompt to perform the entire task, we divide the work into smaller steps. Each step performs one responsibility, stores its result in a shared state, and passes control to the next step.

## What students will learn

- What LangGraph is and why it is useful
- How **state** carries data through a workflow
- How **nodes** perform individual tasks
- How **edges** control the direction of execution
- How to build sequential, parallel, conditional, and looping workflows
- How multiple AI steps can work together like a team of specialized agents


In [1]:
!pip install -U langgraph langchain langchain-core langchain-community \
               langchain-openai langchain-experimental \
               pydantic typing-extensions

  Using cached langgraph-1.2.10-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain-1.3.14-py3-none-any.whl.metadata (6.1 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pydantic_core-2.46.4-cp314-cp314-win_amd64.whl.metadata (6.7 kB)
Using cached langgraph-1.2.10-py3-none-any.whl (247 kB)
   ---------------------------------------- 0.0/561.7 kB ? eta -:--:--
   ---------------------------------------- 561.7/561.7 kB 9.4 MB/s  0:00:00
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.4-cp314-cp314-win_amd64.whl (2.1 MB)
Using cached langchain-1.3.14-py3-none-any.whl (139 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)

  Attempting uninstall: pydantic-core

    Found existing installation: pydantic_core 2.41.5

    Uninstalling pydantic_core-2.41.5:

      Successfully uninstalled pydantic_core-2.41.5

   ----

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openrouter 0.11.46 requires pydantic<2.13,>=2.11.2, but you have pydantic 2.13.4 which is incompatible.


In [2]:
# -----------------------------------------
# Imports
# -----------------------------------------
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
import os

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage



In [3]:
import os
import getpass

os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OpenRouter API Key: ")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


## 1. Shared State: The Workflow's Memory

A LangGraph workflow needs a common data structure called **state**.

Think of the state as a shared notebook carried through the graph. Every node can:

1. Read information already stored in the state
2. Perform its task
3. Return one or more updated state values

In this example, the state begins with a product name. As the graph runs, each node adds new information such as the basic description, features, marketing message, and final description.

```text
Initial State
    ↓
product_name = "Smart Water Bottle"
    ↓
Nodes gradually fill the remaining fields
```

`TypedDict` describes which keys are expected in the state and the type of value stored under each key.


In [8]:
# -----------------------------------------
# LLM (OpenRouter)
# -----------------------------------------
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.3,
    max_tokens = 300,
    openai_api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

# -----------------------------------------
# Graph State
# -----------------------------------------
class State(TypedDict):
    product_name: str
    basic_description: str
    features_benefits: str
    marketing_message: str
    final_description: str



## 2. Nodes: The Workers of the Graph

A **node** is a Python function that performs one specific step in the workflow.

Each node:

- Receives the current state
- Reads the data it needs
- Performs an operation, such as calling an LLM
- Returns a dictionary containing updated state fields

For example:

```python
def generate_basic_description(state: State):
    ...
    return {"basic_description": response.content}
```

This node reads `product_name` and writes its result into `basic_description`.

| Node name | Python function | Responsibility |
|---|---|---|
| `basic` | `generate_basic_description` | Creates the first product description |
| `features` | `add_features_benefits` | Extracts features and benefits |
| `marketing` | `create_marketing_message` | Produces persuasive marketing copy |
| `final` | `polish_final_description` | Produces the polished final output |

The node name is the label used inside the graph. The function contains the actual task logic.


In [5]:
# -----------------------------------------
# Nodes
# -----------------------------------------

def generate_basic_description(state: State):
    response = llm.invoke([
      SystemMessage(content="You are a helpful assistant that generates brief product descriptions."),
        HumanMessage(content=f"Write a brief description of a product named '{state['product_name']}'.")
    ])
    return {"basic_description": response.content}


def add_features_benefits(state: State):
    response = llm.invoke([
        HumanMessage(content=f"List key features and benefits of the product:\n{state['basic_description']}")
    ])
    return {"features_benefits": response.content}


def create_marketing_message(state: State):
    response = llm.invoke([
        HumanMessage(content=f"Create a compelling marketing message using:\n{state['features_benefits']}")
    ])
    return {"marketing_message": response.content}


def polish_final_description(state: State):
    response = llm.invoke([
        HumanMessage(content=f"""
Polish and finalize the product description using the marketing message below:

{state['marketing_message']}
""")
    ])
    return {"final_description": response.content}



## 3. Edges: The Routes Between Nodes

An **edge** tells LangGraph which node should run next.

Think of nodes as cities and edges as roads connecting them.

```text
START → basic → features → marketing → final → END
```

### Normal edge

```python
workflow.add_edge("basic", "features")
```

After `basic` finishes, LangGraph runs `features`.

### START edge

```python
workflow.add_edge(START, "basic")
```

`START` is the special entry point of the graph.

### END edge

```python
workflow.add_edge("final", END)
```

`END` tells LangGraph to stop and return the final state.

### Building the workflow

- `StateGraph(State)` creates a graph using the defined state
- `add_node()` registers a function as a graph step
- `add_edge()` connects graph steps
- `compile()` validates the graph and makes it executable


In [6]:
# -----------------------------------------
# Build LangGraph Workflow
# -----------------------------------------
def build_workflow():
    workflow = StateGraph(State)

    workflow.add_node("basic", generate_basic_description)
    workflow.add_node("features", add_features_benefits)
    workflow.add_node("marketing", create_marketing_message)
    workflow.add_node("final", polish_final_description)

    workflow.add_edge(START, "basic")
    workflow.add_edge("basic", "features")
    workflow.add_edge("features", "marketing")
    workflow.add_edge("marketing", "final")
    workflow.add_edge("final", END)

    return workflow.compile()



## 4. Running the Sequential Graph

`app.invoke(initial_state)` starts the graph.

LangGraph then:

1. Sends the initial state to `basic`
2. Merges the returned update into the shared state
3. Follows the edge to the next node
4. Continues until it reaches `END`
5. Returns the completed state as `result`

A node normally returns only the fields it changes. LangGraph merges those updates into the existing state.


In [7]:
# -----------------------------------------
# Run
# -----------------------------------------
if __name__ == "__main__":
    app = build_workflow()

    initial_state: State = {
        "product_name": "Smart Water Bottle",
        "basic_description": "",
        "features_benefits": "",
        "marketing_message": "",
        "final_description": "",
    }

    result = app.invoke(initial_state)

    print("\n--- BASIC DESCRIPTION ---\n")
    print(result["basic_description"])

    print("\n--- FEATURES & BENEFITS ---\n")
    print(result["features_benefits"])

    print("\n--- MARKETING MESSAGE ---\n")
    print(result["marketing_message"])

    print("\n--- FINAL DESCRIPTION ---\n")
    print(result["final_description"])


--- BASIC DESCRIPTION ---

The Smart Water Bottle is an innovative hydration solution designed to keep you on track with your daily water intake. Featuring a sleek, ergonomic design, this bottle syncs with a mobile app to monitor your hydration levels and send reminders to drink water throughout the day. Made from durable, BPA-free materials, it includes a built-in LED display that shows your progress and can even track your fitness activities. With its leak-proof lid and easy-to-clean design, the Smart Water Bottle is perfect for the gym, office, or on-the-go hydration. Stay healthy and hydrated effortlessly!

--- FEATURES & BENEFITS ---

### Key Features of the Smart Water Bottle:

1. **Mobile App Integration**: Syncs with a dedicated mobile app to monitor hydration levels and track daily water intake.
  
2. **Reminders**: Sends notifications and reminders to encourage regular water consumption throughout the day.

3. **Ergonomic Design**: Sleek and comfortable design for easy handl

# Advanced LangGraph Flow: Parallel Branches and a Loop

The second workflow demonstrates:

- **Fan-out:** one node sends work to multiple nodes
- **Parallel processing:** independent nodes handle separate tasks
- **Fan-in:** multiple branches join at one node
- **Conditional edge:** the next route depends on state
- **Loop:** execution returns to an earlier step for improvement

```text
                         ┌→ features ─┐
START → basic description├→ audience ─┼→ marketing → final → evaluate
                         └→ SEO ──────┘                    │
                                                          ├→ END
                                                          └→ improve → evaluate
```


In [9]:
# -----------------------------------------
# Imports
# -----------------------------------------
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
import os
import random

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# -----------------------------------------
# Load API key
# -----------------------------------------
import os
import getpass

os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OpenRouter API Key: ")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.4,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)



## 5. Parallel Nodes and Fan-Out

After the basic description is created, three independent tasks begin:

- Generate features and benefits
- Identify the target audience
- Generate SEO keywords

```python
workflow.add_edge("basic", "features")
workflow.add_edge("basic", "audience")
workflow.add_edge("basic", "seo")
```

This one-to-many branching pattern is called **fan-out**.

Each branch updates a different state field:

```text
features node  → features_benefits
audience node  → target_audience
seo node       → seo_keywords
```

Parallel nodes should generally avoid writing to the same state key unless a reducer or merge strategy is configured.

## 6. Fan-In and the Merge Node

The `marketing` node uses the outputs of all three branches.

```text
features ─┐
audience ─┼→ marketing
SEO ──────┘
```

This many-to-one pattern is called **fan-in**. The shared state allows the marketing node to read all three results together.


In [10]:
# -----------------------------------------
# State
# -----------------------------------------
class State(TypedDict):
    product_name: str
    basic_description: str
    features_benefits: str
    target_audience: str
    seo_keywords: str
    marketing_message: str
    final_description: str
    quality_score: int

# -----------------------------------------
# Nodes
# -----------------------------------------

def generate_basic_description(state: State):
    response = llm.invoke([
        SystemMessage(content="You generate short product descriptions."),
        HumanMessage(content=f"Write a brief description of '{state['product_name']}'.")
    ])
    return {"basic_description": response.content}


# ----------- PARALLEL NODES -----------

def add_features(state: State):
    response = llm.invoke(
        f"List key features and benefits:\n{state['basic_description']}"
    )
    return {"features_benefits": response.content}


def identify_audience(state: State):
    response = llm.invoke(
        f"Who is the ideal target audience for this product?\n{state['basic_description']}"
    )
    return {"target_audience": response.content}


def generate_seo_keywords(state: State):
    response = llm.invoke(
        f"Generate SEO keywords for this product:\n{state['basic_description']}"
    )
    return {"seo_keywords": response.content}


# ----------- MERGE NODE -----------

def create_marketing_message(state: State):
    combined = f"""
Features:
{state['features_benefits']}

Audience:
{state['target_audience']}

SEO:
{state['seo_keywords']}
"""
    response = llm.invoke(
        f"Create a compelling marketing message using:\n{combined}"
    )
    return {"marketing_message": response.content}





## 7. Conditional Edges and Loops

A normal edge always follows the same route. A **conditional edge** chooses the next route by checking the state.

```python
lambda state: "improve" if state["quality_score"] < 6 else "end"
```

This means:

```text
score below 6 → improve the description
score 6 or above → finish
```

The improvement node connects back to evaluation:

```python
workflow.add_edge("improve", "evaluate")
```

This creates a loop:

```text
evaluate → improve → evaluate
```

Loops are useful for reflection, correction, validation, retries, and approval workflows.

> This notebook uses a random quality score for demonstration. A real application could use an LLM evaluator, business rules, a validation model, or human feedback. Production loops should also have a maximum retry count.


In [11]:
# ----------- FINAL POLISH -----------

def polish_final_description(state: State):
    response = llm.invoke(
        f"Polish and finalize:\n{state['marketing_message']}"
    )
    return {"final_description": response.content}


# ----------- QUALITY CHECK (LOOP CONTROL) -----------

def evaluate_quality(state: State):
    """
    Simulated quality scoring.
    In real systems you'd use LLM scoring.
    """
    score = random.randint(5, 7)
    print(f"\nQuality Score: {score}")
    return {"quality_score": score}


def improve_description(state: State):
    response = llm.invoke(
        f"Improve this product description to make it more persuasive:\n{state['final_description']}"
    )
    return {"final_description": response.content}




## 8. Reading the Advanced Graph Definition

The graph is created in four stages:

1. Register nodes with `add_node()`
2. Connect `START` to the first node
3. Create parallel branches and merge them
4. Add conditional routing and the improvement loop

A fixed route uses:

```python
workflow.add_edge("marketing", "final")
```

A dynamic route uses:

```python
workflow.add_conditional_edges("evaluate", routing_function, route_map)
```

The routing function inspects the state and returns a route name. The route map converts that name into a node or `END`.


In [12]:

# -----------------------------------------
# Build Graph
# -----------------------------------------
def build_workflow():
    workflow = StateGraph(State)
    # Nodes
    workflow.add_node("basic", generate_basic_description)
    workflow.add_node("features", add_features)
    workflow.add_node("audience", identify_audience)
    workflow.add_node("seo", generate_seo_keywords)
    workflow.add_node("marketing", create_marketing_message)
    workflow.add_node("final", polish_final_description)
    workflow.add_node("evaluate", evaluate_quality)
    workflow.add_node("improve", improve_description)
    # Flow
    workflow.add_edge(START, "basic")
    # ---- PARALLEL FAN OUT ----
    workflow.add_edge("basic", "features")
    workflow.add_edge("basic", "audience")
    workflow.add_edge("basic", "seo")
    # ---- FAN IN (merge) ----
    workflow.add_edge("features", "marketing")
    workflow.add_edge("audience", "marketing")
    workflow.add_edge("seo", "marketing")
    workflow.add_edge("marketing", "final")
    # ---- LOOP SECTION ----
    workflow.add_edge("final", "evaluate")
    workflow.add_conditional_edges(
        "evaluate",
        lambda state: "improve" if state["quality_score"] < 6 else "end",
        {
            "improve": "improve",
            "end": END
        }
    )
    workflow.add_edge("improve", "evaluate")
    return workflow.compile()


## 9. How the State Grows

For `"Smart Water Bottle"`, the state changes approximately like this:

```text
Step 1: product_name
Step 2: + basic_description
Step 3: + features_benefits
        + target_audience
        + seo_keywords
Step 4: + marketing_message
Step 5: + final_description
Step 6: + quality_score
Step 7: improve and evaluate again, or stop
```

The final `result` contains the complete state when the graph reaches `END`.


In [ ]:
# -----------------------------------------
# Run
# -----------------------------------------
if __name__ == "__main__":
    app = build_workflow()

    initial_state: State = {
        "product_name": "Smart Water Bottle",
        "basic_description": "",
        "features_benefits": "",
        "target_audience": "",
        "seo_keywords": "",
        "marketing_message": "",
        "final_description": "",
        "quality_score": 0
    }

    result = app.invoke(initial_state)

    print("\nFINAL OUTPUT:\n")
    print(result["final_description"])


Quality Score: 7

FINAL OUTPUT:

**Stay Hydrated, Stay Smart: Meet Your New Favorite Companion!**

Introducing the **Smart Water Bottle**—the ultimate hydration solution for the modern lifestyle. Whether you’re a fitness enthusiast, a busy professional, or an eco-conscious consumer, our innovative water bottle is here to transform the way you hydrate.

### **Why Choose the Smart Water Bottle?**

**💧 Water Intake Tracking**: Never lose sight of your hydration goals again! Our built-in tracker monitors and records your daily water consumption, ensuring you stay on top of your health.

**📱 Smartphone Syncing**: Connect effortlessly to our user-friendly app for real-time hydration stats and personalized goals right at your fingertips.

**🔔 Customizable Reminders**: Say goodbye to dehydration! Receive timely alerts that remind you to drink water throughout your busy day.

**🌡️ Temperature Control**: Enjoy your favorite beverages at the perfect temperature. Whether it’s hot coffee or ice-co


Quality Score: 6


## 10. Gradio Interface

The Gradio section provides a visual interface around the same compiled LangGraph workflow.

When a user enters a product name:

1. Gradio creates the initial state
2. `app.invoke()` runs the graph
3. LangGraph returns the final state
4. Gradio displays the output produced by each node

This helps students observe how a single input is transformed step by step.


In [ ]:
# =========================================================
# GRADIO UI FOR SECOND LANGGRAPH AGENTIC WORKFLOW
# Shows output of each agent/node step
# =========================================================

!pip install -q gradio

import gradio as gr

# Build the already-defined workflow
app = build_workflow()


def run_product_agent(product_name):
    initial_state: State = {
        "product_name": product_name,
        "basic_description": "",
        "features_benefits": "",
        "target_audience": "",
        "seo_keywords": "",
        "marketing_message": "",
        "final_description": "",
        "quality_score": 0
    }

    result = app.invoke(initial_state)

    basic = result.get("basic_description", "")
    features = result.get("features_benefits", "")
    audience = result.get("target_audience", "")
    seo = result.get("seo_keywords", "")
    marketing = result.get("marketing_message", "")
    final_description = result.get("final_description", "")
    quality_score = result.get("quality_score", "")

    process_view = f"""
# LangGraph Agentic Product Description Process

## 1. Basic Description Agent
{basic}

---

## 2. Features & Benefits Agent
{features}

---

## 3. Target Audience Agent
{audience}

---

## 4. SEO Keywords Agent
{seo}

---

## 5. Marketing Message Agent
{marketing}

---

## 6. Final Polish Agent
{final_description}

---

## 7. Quality Evaluation Agent
**Quality Score:** {quality_score}

---

## Final Output
{final_description}
"""

    return (
        basic,
        features,
        audience,
        seo,
        marketing,
        str(quality_score),
        final_description,
        process_view
    )


with gr.Blocks(title="LangGraph Product Description Agent") as demo:
    gr.Markdown("# LangGraph Product Description Agent")
    gr.Markdown(
        "Enter a product name and see how each LangGraph agent transforms it step by step."
    )

    with gr.Row():
        product_input = gr.Textbox(
            label="Product Name",
            value="Smart Water Bottle",
            placeholder="Enter product name..."
        )

    run_button = gr.Button("Generate Product Description")

    with gr.Tab("Step-by-Step Agent Outputs"):
        basic_output = gr.Textbox(
            label="1. Basic Description Agent",
            lines=6
        )

        features_output = gr.Textbox(
            label="2. Features & Benefits Agent",
            lines=8
        )

        audience_output = gr.Textbox(
            label="3. Target Audience Agent",
            lines=8
        )

        seo_output = gr.Textbox(
            label="4. SEO Keywords Agent",
            lines=6
        )

        marketing_output = gr.Textbox(
            label="5. Marketing Message Agent",
            lines=8
        )

        quality_output = gr.Textbox(
            label="6. Quality Score"
        )

        final_output = gr.Textbox(
            label="7. Final Product Description",
            lines=10
        )

    with gr.Tab("Full Process View"):
        full_process_output = gr.Markdown()

    run_button.click(
        fn=run_product_agent,
        inputs=product_input,
        outputs=[
            basic_output,
            features_output,
            audience_output,
            seo_output,
            marketing_output,
            quality_output,
            final_output,
            full_process_output
        ]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.



Quality Score: 5

Quality Score: 6
